# Chapter 08 Companion Notebook: SVM: Bank Customer Classification

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch08_SVM_Bank_Customer.ipynb)

This notebook accompanies Chapter 08 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
- Click "Upload" and select this file and the data file.
- https://archive.ics.uci.edu/dataset/222/bank+marketing

# Bank customer: SVM

### Use "Bank customer.csv"
Build a classification algorithm for bank customers' response to marketing campaign.
- age (numeric)  
- marital: marital status (categorical: "married", "divorced", "single"; "divorced" means divorced or widowed)  - education (categorical: "secondary", "primary", "tertiary")  
- default: has credit in default? (binary: "yes", "no")  
- balance: average yearly balance, in euros (numeric)  
- housing: has housing loan? (binary: "yes", "no")  
- loan: has personal loan? (binary: "yes", "no")  
- duration: last contact duration, in seconds (numeric)  
- campaign: number of contacts performed during this campaign (numeric, includes last contact)  
- pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric)
- previous: number of contacts performed before this campaign (numeric)  
- poutcome: outcome of the previous marketing campaign (categorical: "failure", "success")
- deposit: has the client subscribed a term deposit? (binary: "yes", "no")

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
# Read the data

df = pd.read_csv('Bank customer.csv')
df.head()

In [ ]:
# Create dummy variables for categorical columns

df = pd.get_dummies(df, drop_first=True)
df.head()

In [ ]:
# Define x and y. Split into train, test data

y=df.deposit_yes
x=df.drop('deposit_yes', axis=1)
xtrain, xtest, ytrain, ytest = train_test_split(x, y, random_state=1)

In [ ]:
# Standardize the training and test data

scaler = StandardScaler()
xtrain = scaler.fit_transform(xtrain)
xtest = scaler.transform(xtest)

### SVM

In [ ]:
# RBF kernel

# Train an SVM model using the RBF kernel
svm_rbf = SVC(kernel='rbf', C=1)
svm_rbf.fit(xtrain, ytrain)

# Predict the class labels for the test data
pred1 = svm_rbf.predict(xtest)

# Evaluate the model
print("RBF Kernel Accuracy:", accuracy_score(ytest, pred1))
print("RBF Kernel Confusion Matrix\n", confusion_matrix(ytest, pred1))
print("RBF Kernel Classification Report\n", classification_report(ytest, pred1))

- `SVC(kernel='rbf', C=1)` initializes a Support Vector Classifier (SVC) object with an RBF kernel.
    - The `C` parameter controls the regularization strength. Higher values lead to fewer misclassifications on the training data, potentially at the cost of overfitting.

In [ ]:
# Train an SVM model using the sigmoid kernel
svm_sm = SVC(kernel='sigmoid', C=0.1)
svm_sm.fit(xtrain, ytrain)

# Predict the class labels for the test data
pred2 = svm_sm.predict(xtest)

# Evaluate the model
print("Sigmoid Kernel Accuracy:", accuracy_score(ytest, pred2))
print("Sigmoid Kernel Confusion Matrix\n", confusion_matrix(ytest, pred2))
print("Sigmoid Kernel Classification Report\n", classification_report(ytest, pred2))

# Hyperparameter tuning

In [ ]:
# Define the SVC model pipeline
svc = make_pipeline(StandardScaler(), SVC())

# Set up the grid search parameter grid
param = {'svc__C': [1, 10], 'svc__kernel': ['rbf', 'sigmoid', 'poly']}

# Initialize the GridSearchCV object with cross-validation
search = GridSearchCV(svc, param, cv=5, scoring=['accuracy', 'f1'], refit='f1', verbose=2).fit(xtrain, ytrain)

# Print the best parameters and the corresponding cross-validated scores
print("Best parameters:", search.best_params_)
print("Best cross-validated accuracy:", search.cv_results_['mean_test_accuracy'][search.best_index_])
print("Best cross-validated F1 score:", search.best_score_)

- `svc = make_pipeline(StandardScaler(), SVC())`
   - `make_pipeline` creates a pipeline. A pipeline is a way to chain multiple processing steps together, such as feature scaling, feature selection, or model training, into a single object. This makes it easier to work with machine learning workflows by encapsulating all the necessary steps into one entity.
   - `StandardScaler()` is for feature scaling.
   - `SVC()` is for Support Vector Classification.
- The `param` dictionary defines the hyperparameters to search over.
   - Each hyperparameter is prefixed with `svc__` to specify that it belongs to the SVC model in the pipeline.
- `search = GridSearchCV(svc, param, cv=5, scoring='accuracy', verbose=2)`
    - `estimator`: The SVC model pipeline (`svc`).
    - `param`: The parameter grid to search over.
    - `cv=5`: 5 fold cross-validation.
    - `scoring`: The evaluation metric used.
    - `refit='f1'` indicates that the GridSearchCV should use the F1 score to select the best model configuration. After the grid search is complete, the estimator with the highest F1 score on the validation set will be retrained on the entire training set.
    - `verbose`: Controls the verbosity of the output during the grid search (set to 2 for detailed output).
- The code loops through each set of parameters and their corresponding metrics, then retrieves the cross-validation results (`cv_results_`) containing evaluation metrics for each set of hyperparameters. For each set of parameters:
    - `search.best_params_` returns the hyperparameters that resulted in the best performance during the grid search.
    - `search.best_score_` returns the mean cross-validated score (F1 in this case) achieved by the best estimator on the validation set
    - `cv_results_` contains a dictionary with detailed information about the cross-validation results for each combination of hyperparameters.
      -  `cv_results_['mean_test_accuracy']` contains the mean test accuracy for each combination of hyperparameters.
      - `search.best_index_` refers to the index in the cv_results_ array that corresponds to the best performing estimator.